In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
from neuro import config
from os.path import join
sys.path.append(join(config.REPO_DIR, 'experiments'))

import dvu
import seaborn as sns
import os
import pandas as pd
from copy import deepcopy
from matplotlib import pyplot as plt
import numpy as np
from neuro import config
import imodelsx.process_results
import neuro.features.qa_questions as qa_questions
import joblib
from tqdm import tqdm
import neuro.viz
from neuro import analyze_helper, viz
fit_encoding = __import__('02_fit_encoding')
dvu.set_style()

# results_dir = config.BEST_RESULTS_DIR_ENSEMBLE
# rr, cols_varied, mets = analyze_helper.load_clean_results(results_dir)
# joblib.dump({'r': rr, 'cols_varied': cols_varied, 'mets': mets}, 'results_best_ensemble.pkl')
data = joblib.load(join(config.RESULTS_DIR_LOCAL, 'results_best_ensemble.pkl'))
rr, cols_varied, mets = data['r'], data['cols_varied'], data['mets']
metric_sort = 'corrs_tune_pc_weighted_mean'

In [ ]:
subject = 'S02'
r = rr[rr.subject.isin([subject])]
r = r[r.feature_space == 'meta-llama/Meta-Llama-3-70B']
r = r[r.num_stories == -1]
# r = r.groupby(['subject']).apply(lambda x: x.sort_values('corrs_test_mean', ascending=False).head(1)).reset_index(drop=True)

In [ ]:
metric_sort = 'corrs_tune_pc_weighted_mean'
args = r.sort_values(metric_sort, ascending=False).iloc[0]

In [ ]:
model_params = joblib.load(
    join(args.save_dir_unique, 'model_params.pkl'))
print(args.feature_space, args.pc_components, args.ndelays)

In [ ]:
wt = model_params['weights'] # wt is (n_delays x n_features) x n_voxels
n_features = wt.shape[0] / args.ndelays
wt = wt.reshape(args.ndelays, int(n_features), -1)
wt = wt.mean(axis=0) # average over delays
# preds_test = stim_test_delayed @ wt
# # stim_test_delayed: np.ndarray
#         n_time_points x(n_delays x n_features)

In [ ]:
wt.shape

In [ ]:
# get embs from text (assume constant wordrate)
text = 'the quick brown fox jumped over the lazy dog'
ngrams_list = imodelsx.util.generate_ngrams_list(text, ngrams=10, pad_starting_ngrams=True)

In [ ]:
lm_emb = imodelsx.llm.LLMEmbs(checkpoint=args.feature_space)

In [ ]:
embs = lm_emb(ngrams_list, layer_idx=args.embedding_layer, batch_size=16)

In [ ]:
embs.shape # n_delays x n_features

# make prediction with wt which is (n_delays x n_features) x n_voxels
preds_all = embs[-args.ndelays:].flatten() @ wt  # n_voxels

# Generate stories

In [16]:
# load llama 3 70B
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "meta-llama/Meta-Llama-3-70B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    # offload_folder="offload",
).eval()

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [24]:
# ----------------------------------------------------------
# Hook: add a vector to the LAST TOKEN hidden state at layer
# ----------------------------------------------------------
def add_vector_to_last_token_hook(vec: torch.Tensor):
    """
    Returns a forward hook that adds `vec` to hidden_states[:, -1, :]
    Works whether module output is a Tensor or a tuple whose first item is Tensor.
    """
    def hook(module, inputs, output):
        print('hooked 2!')
        # `output` for LlamaDecoderLayer is typically a tuple:
        # (hidden_states, present_key_value, ...) depending on config/use_cache
        if isinstance(output, tuple):
            hidden = output[0]
        else:
            hidden = output

        # hidden: [batch, seq_len, hidden_size]
        # Add only to the last token position.
        hidden[:, -1, :] += vec
        # print(hidden[:, -1, :20])

        # Return in the same structure HF expects
        if isinstance(output, tuple):
            return (hidden,) + output[1:]
        else:
            return hidden

    return hook

# -----------------------
# Choose layer + vector
# -----------------------
# IMPORTANT: HF layers are 0-indexed.
LAYER_IDX = 16

# For Llama-style models, decoder blocks are usually here:
# model.model.layers[LAYER_IDX]
layer = model.model.layers[LAYER_IDX]

hidden_size = model.config.hidden_size

# Your intervention vector (shape [hidden_size]).
# Replace this with your real vector.
# Tip: keep dtype/device aligned with the model output dtype/device.
device = next(model.parameters()).device
dtype = next(model.parameters()).dtype

intervention_vec = torch.zeros(hidden_size, device=device, dtype=dtype)
# Example: set a few dims just to prove it works
intervention_vec[:8] = torch.tensor([10, 100, 100.1, 100.0, 0.3, 0.2, -0.1, 0.4], device=device, dtype=dtype)

# Hook expects broadcastable shape [1, 1, hidden_size]
intervention_vec_b = intervention_vec.view(1, -1)

# -----------------------
# Run generation w/ hook
# -----------------------
prompt = "Write a short paragraph about mechanistic interpretability."

inputs = tokenizer(prompt, return_tensors="pt")
# With device_map="auto", inputs must be on the *first* device used for embeddings.
# A reliable way: push to model's input embedding device.
embed_device = model.get_input_embeddings().weight.device
inputs = {k: v.to(embed_device) for k, v in inputs.items()}

handle = layer.register_forward_hook(add_vector_to_last_token_hook(intervention_vec_b))

try:
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            # temperature=0.8,
            # top_p=0.95,
            use_cache=True,  # hook will fire on prefill + each decode step
            pad_token_id=tokenizer.eos_token_id,
        )
finally:
    handle.remove()

print(tokenizer.decode(out[0], skip_special_tokens=True))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


hooked 2!
hooked 2!
hooked 2!
hooked 2!
hooked 2!
Write a short paragraph about mechanistic interpretability.akovakovakovakovakov
